# 08 — Kantorovich Duality

Every transport LP has a hidden twin: a *dual* that prices positions instead of moving mass. The two views give the same number — and the dual is where the deep structure lives.

**Prerequisites:** `07_mccann_geodesic`.

**What you'll be able to do**
- State the Kantorovich dual LP and verify primal–dual equality (no duality gap).
- See complementary slackness — the plan moves mass only where the dual constraint is tight.
- Recognise the Kantorovich–Rubinstein 1-Lipschitz form of $W_1$.

You can already compute optimal transport. Now meet its hidden twin. Instead of asking *how to move the mass*, the dual asks for a **price function** on positions that certifies the transport cost is right. The two questions return the same number, and the dual is where the structure lives: it survives the entropic regularisation of the next notebook, it powers the Sinkhorn algorithm, and it carries all the way to quantum optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.transport.discrete import squared_euclidean_cost, cityblock_cost

np.random.seed(42)
viz.use_course_style()

## 1. The dual face of the LP

Every linear program has a **dual**. For the Kantorovich primal

$$ \min_{P \ge 0,\ P\mathbf{1} = a,\ P^\top\mathbf{1} = b}\ \langle C, P\rangle, $$

the dual reads

$$ \max_{\varphi \in \mathbb{R}^n,\ \psi \in \mathbb{R}^m}\ \langle a, \varphi\rangle + \langle b, \psi\rangle \quad\text{subject to}\quad \varphi_i + \psi_j \le C_{ij}\ \ \forall\, i, j. $$

The dual variables $\varphi, \psi$ are the **Kantorovich potentials** — a price per unit of mass at each source and target position. By LP duality (both polytopes are bounded, so there is no gap) the dual maximum equals the primal minimum. Let's verify it.

In [ ]:
# A small problem: 4 source atoms, 5 target atoms, squared ground cost.
rng = np.random.default_rng(0)
a = rng.random(4); a = a / a.sum()
b = rng.random(5); b = b / b.sum()
cost = squared_euclidean_cost(np.arange(4, dtype=float), np.linspace(0, 3, 5))

# Solve the primal and read POT's dual potentials from the log.
plan, log_info = ot.emd(a, b, cost, log=True)
phi, psi = log_info["u"], log_info["v"]

primal_value  = float(np.sum(cost * plan))
dual_value    = float(a @ phi + b @ psi)
max_violation = float(np.max(phi[:, None] + psi[None, :] - cost))

print(f"primal   <C, P*>              = {primal_value:.6f}")
print(f"dual     <a, phi> + <b, psi>  = {dual_value:.6f}")
print(f"duality gap                    = {abs(primal_value - dual_value):.2e}    (should be 0)")
print(f"max dual constraint violation  = {max_violation:+.2e}   (should be <= 0)")

**Read the output.** The primal and dual values agree to machine precision — **no duality gap** — and every dual constraint $\varphi_i + \psi_j \le C_{ij}$ holds. The LP is solved twice over, by one number with two stories: the cheapest way to move mass, and the highest consistent price for it.

## 2. Complementary slackness

Duality says more than "same number". The **slack** of a constraint is $s_{ij} = C_{ij} - \varphi_i - \psi_j \ge 0$, and **complementary slackness** demands that mass flows *only* where the slack is zero:

$$ P^\star_{ij} > 0 \ \Longrightarrow\ \varphi_i + \psi_j = C_{ij}. $$

So the support of the optimal plan must sit exactly on the tight (zero-slack) cells. Let's see it.

In [ ]:
slack = cost - (phi[:, None] + psi[None, :])   # C_ij - phi_i - psi_j  >= 0
support = plan > 1e-9
rows, cols = np.where(support)

print(f"max slack over all cells:            {slack.max():.4f}")
print(f"max slack on the plan's support:     {slack[support].max():.2e}   (-> ~0 by complementary slackness)")

fig, ax = plt.subplots(figsize=(6.6, 5))
im = ax.imshow(slack, cmap=viz.CMAP_COST, origin="lower")
ax.scatter(cols, rows, marker="o", s=160, facecolors="none",
           edgecolors=viz.COLORS["highlight"], linewidths=2.4, label="optimal plan support")
ax.set_xlabel("target j")
ax.set_ylabel("source i")
ax.set_title("Dual slack  C_ij - phi_i - psi_j   (rings = where P* > 0)", pad=12)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.85, label="slack (>= 0)")
ax.legend(loc="upper right")
plt.show()

**Read the figure.** The heatmap is the dual slack — light cells are expensive detours the dual has priced out. Every ring (a cell where the optimal plan actually moves mass) sits on a **dark, zero-slack** cell. That is complementary slackness made visible: the plan and the prices are locked together, mass moving only along routes the dual certifies as exactly fair.

## 3. The Kantorovich–Rubinstein form of $W_1$

For the ground cost $c(x, y) = |x - y|$, the dual takes a famous shape. The constraint $\varphi(x) - \varphi(y) \le |x - y|$ is exactly the **1-Lipschitz** condition, and the dual collapses to a maximisation over a *single* test function:

$$ W_1(\mu, \nu) = \sup_{\mathrm{Lip}(f) \le 1} \ \int f\,\mathrm{d}\mu - \int f\,\mathrm{d}\nu. $$

A 1-Lipschitz $f$ scores distributions by how far they push mass along the line. This is the form most used in practice (and the one that reappears in the Wasserstein GAN). Let's confirm the potential POT returns is 1-Lipschitz and reproduces $W_1$.

In [ ]:
positions = np.linspace(0.0, 5.0, 6)
rng2 = np.random.default_rng(3)
mu = rng2.random(6); mu /= mu.sum()
nu = rng2.random(6); nu /= nu.sum()

cost_l1 = cityblock_cost(positions, positions)         # c(x, y) = |x - y|
w1 = float(ot.emd2(mu, nu, cost_l1))
_, log_l1 = ot.emd(mu, nu, cost_l1, log=True)
f, g = log_l1["u"], log_l1["v"]
dual_w1 = float(mu @ f + nu @ g)

ratios = [abs(f[i] - f[j]) / abs(positions[i] - positions[j])
          for i in range(6) for j in range(6) if i != j]

print(f"W1 (primal, ot.emd2)         = {w1:.4f}")
print(f"dual value <mu, f> + <nu, g> = {dual_w1:.4f}")
print(f"max Lipschitz ratio of f      = {max(ratios):.4f}   (Kantorovich-Rubinstein: <= 1)")

**Read the output.** The dual value reproduces $W_1$, and the potential's largest slope ratio $|f_i - f_j| / |x_i - x_j|$ stays at or below $1$ — it is a genuine 1-Lipschitz test function. So $W_1$ is both *the least cost to move the mass* and *the most a unit-slope ruler can tell the two distributions apart*. Two faces, one distance.

## Your turn

1. **Offset freedom.** Add a constant $c$ to every $\varphi_i$ and subtract it from every $\psi_j$. Show the dual value $\langle a, \varphi\rangle + \langle b, \psi\rangle$ is unchanged (because $\sum_i a_i = \sum_j b_j = 1$). Potentials are defined only up to this shift.
2. **Count the tight constraints.** How many cells have near-zero slack? Compare that count to the size of the optimal plan's support and to the vertex bound $n + m - 1$.
3. **Build a 1-Lipschitz ruler by hand.** On the six positions of §3, design your own 1-Lipschitz $f$ (slopes in $[-1, 1]$) and compute $\int f\,\mathrm{d}\mu - \int f\,\mathrm{d}\nu$. How close to $W_1$ can you push it before violating the slope bound?

## What you built

- You stated the Kantorovich dual LP and verified primal–dual equality — no duality gap.
- You saw complementary slackness: the optimal plan moves mass only where the dual constraint is tight.
- You met the Kantorovich–Rubinstein 1-Lipschitz form of $W_1$ and confirmed it numerically.

You now hold both faces of optimal transport — the plan and the prices. The dual is not a curiosity: the next three notebooks build the fast Sinkhorn solver and Amari's bridge directly on this structure.

## What's next

The LP is exact but costs $O(n^3)$ — impractical at data scale. In `09_entropic_regularization` we add an entropy bonus that makes the problem strongly convex and smooth, and reveals the **Gibbs kernel** at the heart of the fast algorithm to come.

## References

- L. V. Kantorovich & G. Rubinstein, "On a space of completely additive functions", *Vestnik Leningrad Univ.* 13, 52–59 (1958).
- C. Villani, *Topics in Optimal Transportation*, AMS (2003), chs. 1 and 5.
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), chs. 2–3. DOI:10.1561/2200000073

**Previous:** `notebooks/03_ClassicalOptimalTransport/07_mccann_geodesic.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/09_entropic_regularization.ipynb`